# End2Race PPO next experiment roadmap

## tl;dr

Cap clip exploration at 0.25. Run clean 45-update clip 0.20/0.25 arms to close both the clip boundary
and training-horizon questions. Freeze gamma because it also changes risk-potential reward; defer GAE
until advantage telemetry is available. Test hard neighbors later as a separate fixed-cache ablation.


## Context & Methods

The decision uses the two clean 12-worker, 30-update Group 5 runs. Austin600 results are compared
at U1/U5/U10/U15/U20/U25/U30. Collision identities are paired by scenario; training telemetry uses
formal rows U2-U30 to reduce the shared U1 warm-up transient.

### Key Assumptions

- Austin600 is a fixed selection panel, not a new IID sample at every checkpoint.
- Exact paired p-values are unadjusted for checkpoint selection and do not establish cross-seed generalization.
- Clip fraction measures how often the recorded importance ratio is outside the configured PPO interval;
  it is not a direct forecast of the effect of a larger clip.


In [1]:
from pathlib import Path
import csv
import json

ROOT = Path(globals().get("_REPO_ROOT", Path.cwd()))
OUT = ROOT / "analysis_results" / "clip_extension_decision"
summary = json.loads((OUT / "decision_summary.json").read_text())
with (OUT / "checkpoint_eval.csv").open() as handle:
    checkpoint_eval = list(csv.DictReader(handle))
with (OUT / "paired_by_checkpoint.csv").open() as handle:
    paired = list(csv.DictReader(handle))
with (OUT / "training_telemetry.csv").open() as handle:
    telemetry = list(csv.DictReader(handle))
with (OUT / "late_training_u20_u30.csv").open() as handle:
    late_training = list(csv.DictReader(handle))
with (OUT / "discount_horizons.csv").open() as handle:
    discount_horizons = list(csv.DictReader(handle))
with (OUT / "hard_neighbor_evidence.csv").open() as handle:
    hard_neighbor = list(csv.DictReader(handle))
with (OUT / "experiment_roadmap.csv").open() as handle:
    roadmap = list(csv.DictReader(handle))
with (OUT / "evaluation_panel_audit.csv").open() as handle:
    evaluation_panel_audit = list(csv.DictReader(handle))
with (OUT / "paired_stat_examples.csv").open() as handle:
    paired_stat_examples = list(csv.DictReader(handle))
print(summary["decision"])
print("clip 0.15:", summary["clip015_collision_path"])
print("clip 0.20:", summary["clip020_collision_path"])


Cap clip exploration at 0.25, extend the clean 0.20/0.25 comparison to 45 updates, freeze gamma, defer GAE, then test a fixed boundary-aware collision cache as a separate axis.
clip 0.15: [16, 18, 18, 14, 14, 17, 20]
clip 0.20: [21, 18, 16, 13, 13, 17, 11]


## Data

### 1. Paired checkpoint results

`resolved_by_clip020` counts 0.15 collision scenarios eliminated by 0.20; `created_by_clip020`
counts new collision scenarios introduced by 0.20.


In [2]:
for row in paired:
    print(row)


{'update': '1', 'clip015_collisions': '16', 'clip020_collisions': '21', 'shared': '10', 'resolved_by_clip020': '6', 'created_by_clip020': '11', 'net_collision_change': '5', 'paired_exact_p_unadjusted': '0.332305908203125'}
{'update': '5', 'clip015_collisions': '18', 'clip020_collisions': '18', 'shared': '11', 'resolved_by_clip020': '7', 'created_by_clip020': '7', 'net_collision_change': '0', 'paired_exact_p_unadjusted': '1.0'}
{'update': '10', 'clip015_collisions': '18', 'clip020_collisions': '16', 'shared': '6', 'resolved_by_clip020': '12', 'created_by_clip020': '10', 'net_collision_change': '-2', 'paired_exact_p_unadjusted': '0.8318119049072266'}
{'update': '15', 'clip015_collisions': '14', 'clip020_collisions': '13', 'shared': '6', 'resolved_by_clip020': '8', 'created_by_clip020': '7', 'net_collision_change': '-1', 'paired_exact_p_unadjusted': '1.0'}
{'update': '20', 'clip015_collisions': '14', 'clip020_collisions': '13', 'shared': '7', 'resolved_by_clip020': '7', 'created_by_clip02

## Review corrections: use paired identities, and audit the holdout constructor

The Austin panel is fixed and paired, so a universal independent-binomial `+/-4.2` noise floor or
an `>8 collisions` detection rule is not valid. Evidence depends on resolved versus created scenario
identities. The current 50-startpoint constructor also aliases its last endpoint to the first after
modulo reduction, yielding 49 unique starts and 588 unique physical scenarios out of 600 slots.


In [3]:
print("Paired statistical examples:")
for row in paired_stat_examples:
    print(row)
print("Evaluation panel audit:")
for row in evaluation_panel_audit:
    print(row)


Paired statistical examples:
{'comparison': 'BC -> privilege_gru base U20', 'left_collisions': '22', 'right_collisions': '14', 'net_reduction': '8', 'resolved': '10', 'created': '2', 'discordant_pairs': '12', 'paired_exact_p_unadjusted': '0.03857421875'}
{'comparison': 'long clip0.15 U30 -> long clip0.20 U30', 'left_collisions': '20', 'right_collisions': '11', 'net_reduction': '9', 'resolved': '15', 'created': '6', 'discordant_pairs': '21', 'paired_exact_p_unadjusted': '0.0783538818359375'}
Evaluation panel audit:
{'ego_idx_offset': '0', 'raceline_waypoints': '2096', 'nominal_startpoints': '50', 'unique_effective_startpoints': '49', 'duplicated_startpoint_slots': '1', 'nominal_scenarios': '600', 'unique_physical_scenarios': '588', 'duplicated_physical_scenario_slots': '12', 'effective_start_overlap_with_offset0': '49'}
{'ego_idx_offset': '1', 'raceline_waypoints': '2096', 'nominal_startpoints': '50', 'unique_effective_startpoints': '49', 'duplicated_startpoint_slots': '1', 'nominal_sce

## Results

### 2. The current best result is real as a recorded checkpoint, but not yet a stable trend

U30 improves from 20 to 11 collisions and resolves 15 scenarios while creating 6. The exact paired
p-value is 0.078 before any correction for selecting U30 after looking at seven checkpoints.
Earlier checkpoints are mostly ties or small differences, and clip 0.20 regresses at U25.


In [4]:
u30 = next(row for row in paired if int(row["update"]) == 30)
print("U30 paired comparison:", u30)
print("Mean collisions across seven checkpoints:",
      round(summary["clip015_mean_collisions_all_checkpoints"], 3),
      round(summary["clip020_mean_collisions_all_checkpoints"], 3))


U30 paired comparison: {'update': '30', 'clip015_collisions': '20', 'clip020_collisions': '11', 'shared': '5', 'resolved_by_clip020': '15', 'created_by_clip020': '6', 'net_collision_change': '-9', 'paired_exact_p_unadjusted': '0.0783538818359375'}
Mean collisions across seven checkpoints: 16.714 15.571


### 3. Clip 0.20 is not obviously over-constrained, but its KL tail is already wide

Only about 5.3% of recorded samples are clipped on average at 0.20 over U2-U30. That leaves a
plausible but small region for 0.25 to change directly. At the same time, 15/29 updates already have
`approx_kl_max > 0.5` and five exceed 1.0, so a larger clip should be treated as a safety-sensitive
probe rather than an automatic improvement.


In [5]:
for row in telemetry:
    print(row)


{'clip': 'clip 0.15', 'updates': '29', 'mean_clip_fraction': '0.08366379105766476', 'median_clip_fraction': '0.0740087874874007', 'mean_approx_kl': '0.055570133630978395', 'updates_kl_max_gt_0_5': '13', 'updates_kl_max_gt_1': '4', 'largest_kl_max': '3.094435453414917'}
{'clip': 'clip 0.20', 'updates': '29', 'mean_clip_fraction': '0.05283388218971407', 'median_clip_fraction': '0.044570311409188434', 'mean_approx_kl': '0.06804283189072081', 'updates_kl_max_gt_0_5': '15', 'updates_kl_max_gt_1': '5', 'largest_kl_max': '3.832719564437866'}


### 4. U30 is not an optimization-convergence point

From U20 through U30, actor checkpoint step norms do not shrink toward zero and approximate KL
continues to spike. This supports extending the horizon as a diagnostic, but the eval path
`13 -> 17 -> 11` does not support a monotonic-improvement claim. A 45-update run must be fresh-started
because current checkpoints omit optimizer, RNG, scheduler, and environment-queue state.


In [6]:
for row in late_training:
    print(row)


{'update': '20', 'rollout_policy_update': '19', 'approx_kl_mean': '0.0039527445792919', 'approx_kl_max': '0.0073198662139475346', 'clip_fraction_mean': '0.03625976464445557', 'explained_variance_post': '0.9379833952439158', 'rollout_collision_count': '42', 'rollout_episode_count': '152', 'actor_step_l2': '0.021087262201976886', 'actor_step_relative_l2': '0.00012957042238959283'}
{'update': '21', 'rollout_policy_update': '20', 'approx_kl_mean': '0.4431079248370952', 'approx_kl_max': '3.832719564437866', 'clip_fraction_mean': '0.09170409876969643', 'explained_variance_post': '0.937254497843986', 'rollout_collision_count': '46', 'rollout_episode_count': '152', 'actor_step_l2': '0.011308550996451753', 'actor_step_relative_l2': '6.948525263743788e-05'}
{'update': '22', 'rollout_policy_update': '21', 'approx_kl_mean': '0.055365539603808425', 'approx_kl_max': '0.4447918236255646', 'clip_fraction_mean': '0.0583837873318771', 'explained_variance_post': '0.9055608258168838', 'rollout_collision_c

### 5. Gamma and GAE lambda are not equivalent tuning axes

`gamma=0.999` is used both by PPO returns and by potential-based risk shaping, so changing it also
changes the reward. `gae_lambda` only changes advantage estimation. At 100 Hz, the current
`gamma*lambda=0.994005` gives TD residuals a half-life of about 1.15 seconds. Lowering lambda to 0.99
shortens that to about 0.63 seconds; raising it increases delayed credit but also variance.


In [7]:
for row in discount_horizons:
    print(row)


{'gamma': '0.999', 'gae_lambda': '0.99', 'gamma_times_lambda': '0.98901', 'td_error_half_life_seconds': '0.6272350515610913', 'geometric_horizon_seconds': '0.9099181073703321', 'weight_after_1s': '0.33118318797366136', 'weight_after_2s': '0.10968230399639753', 'weight_after_4s': '0.01203020780995816', 'weight_after_8s': '0.00014472589995077832'}
{'gamma': '0.999', 'gae_lambda': '0.995', 'gamma_times_lambda': '0.994005', 'td_error_half_life_seconds': '1.1527395991034204', 'geometric_horizon_seconds': '1.6680567139282811', 'weight_after_1s': '0.5480963338904562', 'weight_after_2s': '0.30040959122415845', 'weight_after_4s': '0.09024592249946599', 'weight_after_8s': '0.00814432652777962'}
{'gamma': '0.999', 'gae_lambda': '0.9975', 'gamma_times_lambda': '0.9965025000000001', 'td_error_half_life_seconds': '1.978368353430495', 'geometric_horizon_seconds': '2.8591851322373687', 'weight_after_1s': '0.7044322955008794', 'weight_after_2s': '0.49622485894463825', 'weight_after_4s': '0.246239110634

### 6. Hard neighbors are promising, but the 11 probe scenarios are not a production pool

The standalone probe produced 11/18 BC collisions versus a 4.45% global candidate rate, but it
deliberately selected three dense collision families and is not integrated. The full outcome lattice
contains 1,042 collision/other adjacent boundary pairs, supporting a general fixed boundary-aware
cache instead of merging or duplicating the 11 examples.


In [8]:
print(hard_neighbor[0])
print("Experiment order:")
for row in roadmap:
    print(row)


{'base_candidate_count': '10800', 'base_collision_count': '479', 'base_valid_collision_rate': '0.04450018580453363', 'probe_neighbor_count': '18', 'probe_neighbor_collisions': '11', 'probe_neighbor_collision_rate': '0.6111111111111112', 'probe_enrichment': '13.732776617954071', 'outcome_flip_boundary_pairs': '1042', 'boundary_unique_scenarios': '1360', 'boundary_collision_side_scenarios': '446', 'boundary_other_side_scenarios': '914', 'pipeline_integrated': 'False'}
Experiment order:
{'priority': '1', 'axis': 'Training horizon', 'treatment': 'clip 0.20, 45 updates', 'comparator': 'existing clip 0.20 through U30', 'decision': 'Run now as a fresh-start horizon diagnostic; assess U35/U40/U45 late-window mean and scenario churn.'}
{'priority': '1', 'axis': 'Clip boundary', 'treatment': 'clip 0.25, 45 updates', 'comparator': 'clip 0.20, 45 updates under the same seed/source/cache', 'decision': 'Maximum clip probe; do not open 0.30/0.40 unless 0.25 wins without safety regression.'}
{'priorit

## Takeaways

1. Fix or explicitly deduplicate the Austin endpoint alias, then verify shifted-holdout index disjointness.
2. Fresh-start 45-update clip0.20; after exact U1-U30 reproduction, reuse seven existing eval panels and evaluate U35/U40/U45.
3. Either test matched clip0.25 before hard-neighbor, or explicitly freeze clip0.20 and cancel the 0.25 question.
4. Keep gamma at 0.999. Defer lambda until non-RNG-consuming advantage telemetry exists.
5. Implement hard neighbors only after clip/horizon freeze, as a fixed schema-2 boundary-aware cache A/B.
